In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import subprocess, os
#from Utils.utils import visualize_hcube
import h5py
import matplotlib.pyplot as plt
from matplotlib.transforms import Affine2D
import h5py, re, tqdm
from collections import Counter
import tqdm

In [2]:
# Mounting dataset from D: to WSL 
WINDOWS_DRIVE = "D:"   
WINDOWS_PATH  = r"D:\HSI_Dataset\HSI_MSI_HyperResolution"  
LINK_NAME     = "data_external"  

# Mount drive if missing
mnt_path = Path(f"/mnt/{WINDOWS_DRIVE[0].lower()}")
#if not mnt_path.exists():
print(f"[INFO] Mounting {WINDOWS_DRIVE} into {mnt_path} ...")
subprocess.run(["sudo", "mkdir", "-p", str(mnt_path)], check=True)
res = subprocess.run(["sudo", "mount", "-t", "drvfs", WINDOWS_DRIVE, str(mnt_path)], capture_output=True, text=True)
if res.returncode != 0:
    raise RuntimeError(f"Failed to mount {WINDOWS_DRIVE}: {res.stderr}")
#else:
#    print(f"[OK] {mnt_path} already exists")

# Verify dataset path 
dataset_path = Path(str(WINDOWS_PATH).replace("\\", "/").replace(":", "").replace("D", "/mnt/d", 1))
if not dataset_path.exists():
    print(f"[WARN] Dataset not found at {dataset_path}. Let's check what's under /mnt/d:")
    os.system("ls -la /mnt/d")
    raise FileNotFoundError("Fix dataset path above and rerun.")
print(f"[OK] Found dataset: {dataset_path}")

# Create symlink inside project
proj_root = Path.cwd()
link_path = proj_root / LINK_NAME
if link_path.exists() or link_path.is_symlink():
    print(f"[INFO] Removing old link {link_path}")
    link_path.unlink()
link_path.symlink_to(dataset_path, target_is_directory=True)
print(f"[OK] Linked {link_path} -> {dataset_path}")

# Show a few sample files for confirmation
import itertools
exts = {".hdf5", ".h5", ".hdr", ".tif", ".tiff"}
found = list(itertools.islice((p for p in link_path.rglob("*") if p.suffix.lower() in exts), 10))
if found:
    print("Sample files:")
    for f in found: print("  ", f.relative_to(link_path))
else:
    print("No .hdf5/.h5/.hdr/.tif files found yet — check deeper folders.")

[INFO] Mounting D: into /mnt/d ...
[OK] Found dataset: /mnt/d/HSI_Dataset/HSI_MSI_HyperResolution
[INFO] Removing old link /home/isaacmuscat/Grain-Variety-Classification/data_external
[OK] Linked /home/isaacmuscat/Grain-Variety-Classification/data_external -> /mnt/d/HSI_Dataset/HSI_MSI_HyperResolution
Sample files:
   processed/quickrun/calibrated_clean.tif
   raw/calibration/FX10/calibration_1705a_0.hdf5
   raw/calibration/FX10/calibration_1705a_1.hdf5
   raw/calibration/FX10/calibration_1705a_2.hdf5
   raw/calibration/FX10/calibration_1705a_3.hdf5
   raw/calibration/FX10/calibration_1705_0.hdf5
   raw/calibration/FX10/calibration_1705_1.hdf5
   raw/calibration/FX10/calibration_1705_2.hdf5
   raw/calibration/FX10/calibration_1705_3.hdf5
   raw/calibration/Snapshot/processed/calibration_1705a_0/image_0000000003.hdr


In [3]:
ROOT = Path("data_external")  
basepath = ROOT / "raw" / "FX10" 

OUTDIR = basepath.parent.parent / "processed" / "quickrun"  
OUTDIR.mkdir(parents=True, exist_ok=True)

print("ROOT:    ", ROOT.resolve())
print("Basepath:", basepath.resolve())
print("OUTDIR:  ", OUTDIR.resolve())


ROOT:     /mnt/d/HSI_Dataset/HSI_MSI_HyperResolution
Basepath: /mnt/d/HSI_Dataset/HSI_MSI_HyperResolution/raw/FX10
OUTDIR:   /mnt/d/HSI_Dataset/HSI_MSI_HyperResolution/processed/quickrun


In [4]:
## Look for all files in the folder that end with .hdf5 
df = pd.DataFrame({"filepath_FX10": list(Path(f"{basepath}").rglob("**/*.hdf5"))})
## Give the sample name 
df['sample_name'] = df.filepath_FX10.apply(lambda x : x.stem)

df

,filepath_FX10,sample_name
0,data_external/raw/FX10/barley_l0.hdf5,barley_l0
1,data_external/raw/FX10/barley_l1.hdf5,barley_l1
2,data_external/raw/FX10/barley_l2.hdf5,barley_l2
3,data_external/raw/FX10/barley_l3.hdf5,barley_l3
4,data_external/raw/FX10/barley_l4.hdf5,barley_l4
...,...,...
175,data_external/raw/FX10/wheatgrass_s0.hdf5,wheatgrass_s0
176,data_external/raw/FX10/wheatgrass_s1.hdf5,wheatgrass_s1
177,data_external/raw/FX10/wheatgrass_s2.hdf5,wheatgrass_s2
178,data_external/raw/FX10/wheatgrass_s3.hdf5,wheatgrass_s3


In [ ]:
import h5py

# find all .hdf5 files
all_files = list(basepath.rglob("*.hdf5"))
print("Total .hdf5 files found:", len(all_files))

df_files = pd.DataFrame({"filepath_FX10": all_files})
df_files["sample_name"] = df_files["filepath_FX10"].apply(lambda p: p.stem)


def resolve_valid_hdf5(path: Path) -> Path | None:
    """
    Returns:
        - Path to a valid file or None if it fails
    """
    path = Path(path)

    try:
        with h5py.File(path, "r"):
            return path
    except Exception:
        return None

df_files["resolved_path"] = df_files["filepath_FX10"].apply(resolve_valid_hdf5)

df_valid = df_files[df_files["resolved_path"].notna()].copy()
df_invalid = df_files[df_files["resolved_path"].isna()].copy()

print("Valid HDF5 files:", len(df_valid))
print("Invalid/unreadable files:", len(df_invalid))

if not df_invalid.empty:
    print("\nUnreadable sample_names:")
    print(df_invalid["sample_name"].tolist())


Total .hdf5 files found: 180
Valid HDF5 files: 158
Invalid/unreadable files: 22

Unreadable sample_names:
['barley_l3', 'barley_l4', 'barley_m0', 'barley_m2', 'barley_m3', 'barley_m4', 'buckwheat_l0', 'buckwheat_l1', 'buckwheat_l2', 'buckwheat_m3', 'buckwheat_m4', 'corn_l1', 'corn_l3', 'corn_l4', 'mix_l3', 'mix_l4', 'mix_m0', 'pumpkin_s0', 'pumpkin_s3', 'spelt_l4', 'wheatgrass_l2', 'wheatgrass_m4']


In [7]:
def load_cube(path, verbose=False):

    with h5py.File(path, 'r') as f:
        f = h5py.File(path)

        if verbose:
            print("What is inside an h5 file?\n")
            for key in f.keys(): print(f"key:'{key}', \nin which there is '{f[key]}' \
            \nwhich we load in a numpy array of shape: {np.array(f[key]).shape}\n");
        
            print(f"Selected to load cube from file:\n{path}")

        hcube = np.array(f['hypercube'][:,:,:])/10000  #.astype('float32')
        darkref = np.array(f['dark_reference'])/10000 #.astype('float32')
        whiteref = np.array(f['white_reference'])/10000
       
    
        hcube = np.swapaxes(hcube,-1,0).astype('float32')
        hcube = np.fliplr(hcube)
       
        wlens = f['hypercube'].attrs['wavelength_nm']

    return hcube, wlens, darkref, whiteref